# ☕ Coffee Sales Analytics & Business Intelligence
## Complete Data Science Project — EDA + Statistical Testing + Machine Learning

**Student Name:** Samarth Jain  
**Course:** [Course Name]  
**University:** [University Name]  
**Academic Year:** 2024–25  

---

## Section 1 — Project Introduction

This project presents a **complete data science pipeline** applied to a real-world coffee sales transaction dataset.

The pipeline covers:
- Dataset inspection and audit
- Data cleaning and validation
- Feature engineering
- Exploratory Data Analysis with **5 visualizations**
- **5 factual observations** and **5 business insights**
- **3 hypothesis tests** using non-parametric statistics
- **3 data-driven recommendations**
- Machine learning: **Daily Revenue Forecasting** (time-series regression)
- ML model evaluation and feature importance analysis

## Section 2 — Business Problem

Coffee businesses generate high-frequency transaction data but often lack the analytical tools to extract actionable insights from it.

Without proper analysis, businesses cannot answer questions like:
- Which products should be stocked more heavily?
- When should maximum staff be deployed?
- Is revenue growing month-over-month?
- What is a realistic revenue target for the next business day?

This project builds the analytical and predictive infrastructure to answer these questions from actual transaction data.

## Section 3 — Objectives

1. Audit and clean the raw transaction dataset.
2. Identify and document derived columns.
3. Explore product, time, and revenue patterns.
4. Test three business hypotheses statistically.
5. Build a daily revenue forecasting model.
6. Compare multiple regression models rigorously.
7. Explain model decisions using feature importance.
8. Deliver a production-ready Node.js analytics dashboard.

## Section 4 — Dataset Loading

In [ ]:
import os
import sys
import warnings
import json

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# ── Path setup ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)  # coffee-sales-analytics/
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'Coffe_sales.csv')
FIGURES_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_PATH, exist_ok=True)

# Load raw dataset
df_raw = pd.read_csv(DATA_PATH)
print(f'Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
df_raw.head()

## Section 5 — Dataset Audit

In [ ]:
print('=== SHAPE ===')
print(df_raw.shape)

print('\n=== DATA TYPES ===')
print(df_raw.dtypes)

print('\n=== MISSING VALUES ===')
print(df_raw.isnull().sum())
print(f'Total missing: {df_raw.isnull().sum().sum()}')

print('\n=== DUPLICATES ===')
print(f'Exact duplicate rows: {df_raw.duplicated().sum()}')

print('\n=== NUMERICAL SUMMARY ===')
df_raw[['hour_of_day', 'money', 'Weekdaysort', 'Monthsort']].describe()

In [ ]:
print('=== UNIQUE VALUES PER CATEGORICAL COLUMN ===')
cats = ['cash_type', 'coffee_name', 'Time_of_Day', 'Weekday', 'Month_name']
for c in cats:
    vals = sorted(df_raw[c].unique())
    print(f'\n{c} ({len(vals)} unique): {vals}')

print('\n=== MONEY RANGE ===')
print(f'Min: {df_raw["money"].min()}  |  Max: {df_raw["money"].max()}')
print(f'Unique money values: {sorted(df_raw["money"].unique())}')

print('\n=== DATE RANGE ===')
print(f'Min Date: {df_raw["Date"].min()}  |  Max Date: {df_raw["Date"].max()}')

print('\n=== HOUR RANGE ===')
print(f'Min hour: {df_raw["hour_of_day"].min()}  |  Max hour: {df_raw["hour_of_day"].max()}')
print(f'Zero/negative money rows: {(df_raw["money"] <= 0).sum()}')
print(f'Out-of-range hours: {((df_raw["hour_of_day"] < 0) | (df_raw["hour_of_day"] > 23)).sum()}')

In [ ]:
# Derived column mapping table
derived_table = pd.DataFrame([
    ['hour_of_day',  'Time (hour component)',   'Yes — hour matches Time column',    'Yes', 'No (use hour_of_day directly)'],
    ['Time_of_Day',  'hour_of_day',              'Yes — Morning=7-11, Afternoon=12-17, Night=18-22', 'Yes', 'Yes (as categorical feature)'],
    ['Weekday',      'Date (day name)',           'Yes — matches Date.dt.strftime("%a")', 'Yes', 'Yes (encoded)'],
    ['Month_name',   'Date (month name)',         'Yes — matches Date.dt.strftime("%b")', 'Yes', 'Yes (encoded)'],
    ['Weekdaysort',  'Weekday (Mon=1…Sun=7)',     'Yes — consistent numeric sort',   'No (redundant)', 'Yes (direct numeric)'],
    ['Monthsort',    'Month_name (Jan=1…Dec=12)', 'Yes — consistent numeric sort',   'No (redundant)', 'Yes (direct numeric)'],
], columns=['Column', 'Possibly Derived From', 'Verified?', 'Use in EDA', 'Use in ML'])

print('DERIVED COLUMN AUDIT:')
derived_table

## Section 6 — Data Cleaning

In [ ]:
print('=== BEFORE CLEANING ===')
print(f'Rows:          {df_raw.shape[0]}')
print(f'Columns:       {df_raw.shape[1]}')
print(f'Missing total: {df_raw.isnull().sum().sum()}')
print(f'Duplicates:    {df_raw.duplicated().sum()}')
print(f'money dtype:   {df_raw["money"].dtype}')
print(f'Date dtype:    {df_raw["Date"].dtype}')

# ── Cleaning Pipeline ─────────────────────────────────────────────────────
df = df_raw.copy()

# Step 1: Parse Date
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
bad_dates = df['Date'].isna().sum()
print(f'\nBad date parses: {bad_dates}')

# Step 2: Parse Time
df['Time_parsed'] = pd.to_datetime(df['Time'], format='%H:%M:%S.%f', errors='coerce')
bad_times = df['Time_parsed'].isna().sum()
print(f'Bad time parses: {bad_times}')

# Step 3: Remove zero/negative money
bad_money = (df['money'] <= 0).sum()
print(f'Zero/negative money: {bad_money}')
df = df[df['money'] > 0]

# Step 4: Remove out-of-range hours
bad_hours = ((df['hour_of_day'] < 0) | (df['hour_of_day'] > 23)).sum()
print(f'Out-of-range hours: {bad_hours}')

# Step 5: Remove exact duplicates
dupes = df.duplicated().sum()
print(f'Exact duplicates: {dupes}')
df = df.drop_duplicates()

print(f'\n=== AFTER CLEANING ===')
print(f'Rows:          {df.shape[0]}')
print(f'Rows removed:  {df_raw.shape[0] - df.shape[0]}')
print(f'Missing total: {df.isnull().sum().sum()}')
print(f'Duplicates:    {df.duplicated().sum()}')

## Section 7 — Date/Time Processing

In [ ]:
# Create transaction_datetime
df['transaction_datetime'] = df['Date'] + pd.to_timedelta(
    df['Time_parsed'].dt.hour * 3600 + df['Time_parsed'].dt.minute * 60 + df['Time_parsed'].dt.second,
    unit='s'
)

# Verify derived columns
df['hour_verified'] = df['transaction_datetime'].dt.hour
df['weekday_verified'] = df['transaction_datetime'].dt.strftime('%a')
df['month_verified'] = df['transaction_datetime'].dt.strftime('%b')

hour_mismatch = (df['hour_of_day'] != df['hour_verified']).sum()
weekday_mismatch = (df['Weekday'] != df['weekday_verified']).sum()
month_mismatch = (df['Month_name'] != df['month_verified']).sum()

print(f'Hour-of-day mismatches:   {hour_mismatch}')
print(f'Weekday mismatches:       {weekday_mismatch}')
print(f'Month mismatches:         {month_mismatch}')

if hour_mismatch == 0 and weekday_mismatch == 0 and month_mismatch == 0:
    print('\n✓ All derived columns are fully consistent with Date/Time.')

df[['transaction_datetime', 'hour_of_day', 'Weekday', 'Month_name', 'coffee_name', 'money']].head()

## Section 8 — Feature Engineering

In [ ]:
# Detect price tiers automatically
tier1 = {24.0, 28.9, 33.8, 38.7}
tier2 = {18.12, 23.02, 27.92, 32.82, 37.72}
tier3 = {21.06, 25.96, 30.86, 35.76}

def assign_tier(p):
    p = round(float(p), 2)
    if p in tier1: return 1
    elif p in tier2: return 2
    elif p in tier3: return 3
    return 0

df = df.sort_values('transaction_datetime').reset_index(drop=True)
df['price_tier'] = df['money'].apply(assign_tier)
df['is_weekend'] = (df['transaction_datetime'].dt.dayofweek >= 5).astype(int)
df['day_of_month'] = df['transaction_datetime'].dt.day
df['quarter'] = df['transaction_datetime'].dt.quarter

print('Price tier distribution:')
print(df['price_tier'].value_counts().sort_index())
print(f'\nTier 0 (unclassified): {(df["price_tier"] == 0).sum()}')

# Daily aggregation for ML
daily = df.groupby('Date').agg(
    revenue=('money', 'sum'),
    transaction_count=('money', 'count'),
).reset_index().sort_values('Date').reset_index(drop=True)

print(f'\nDaily aggregation shape: {daily.shape}')
print(f'Date range: {daily["Date"].min().date()} → {daily["Date"].max().date()}')
daily.head()

## Section 9 — Exploratory Data Analysis

In [ ]:
# Core summary statistics
total_tx = len(df)
total_rev = df['money'].sum()
avg_tx = df['money'].mean()
med_tx = df['money'].median()
n_products = df['coffee_name'].nunique()
active_days = df['Date'].nunique()
top_coffee = df['coffee_name'].value_counts().idxmax()
peak_hour = df['hour_of_day'].value_counts().idxmax()

print(f'Total Transactions:     {total_tx:,}')
print(f'Total Revenue:          {total_rev:,.2f} CZK')
print(f'Avg Transaction Value:  {avg_tx:.2f} CZK')
print(f'Median Transaction Val: {med_tx:.2f} CZK')
print(f'Coffee Products:        {n_products}')
print(f'Active Sales Days:      {active_days}')
print(f'Top-Selling Coffee:     {top_coffee}')
print(f'Peak Sales Hour:        {peak_hour}:00')

print('\n--- Product Transaction Counts ---')
print(df['coffee_name'].value_counts())

print('\n--- Revenue by Product ---')
print(df.groupby('coffee_name')['money'].sum().sort_values(ascending=False).round(2))

print('\n--- Transactions by Time of Day ---')
print(df['Time_of_Day'].value_counts())

## Section 10 — Exactly 5 Main Visualizations

In [ ]:
# ── Visualization 1: Transaction Count by Coffee Product ─────────────────────
counts = df['coffee_name'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(counts.index, counts.values, color='#3b82d4', edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2, f'{int(w):,}', va='center', ha='left', fontsize=9)
ax.set_xlabel('Number of Transactions', fontsize=11)
ax.set_ylabel('Coffee Product', fontsize=11)
ax.set_title('Visualization 1 — Transaction Count by Coffee Product', fontsize=13, fontweight='bold')
ax.set_xlim(0, counts.max() * 1.15)
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'viz1_product_transactions.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Top product: {counts.index[-1]} ({counts.values[-1]:,} transactions)')

In [ ]:
# ── Visualization 2: Revenue by Coffee Product ───────────────────────────────
rev = df.groupby('coffee_name')['money'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(rev.index, rev.values, color='#7c5cd8', edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 50, bar.get_y() + bar.get_height()/2, f'{w:,.0f}', va='center', ha='left', fontsize=9)
ax.set_xlabel('Total Revenue (CZK)', fontsize=11)
ax.set_ylabel('Coffee Product', fontsize=11)
ax.set_title('Visualization 2 — Total Revenue by Coffee Product', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlim(0, rev.max() * 1.18)
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'viz2_product_revenue.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Top revenue product: {rev.index[-1]} ({rev.values[-1]:,.2f} CZK)')

In [ ]:
# ── Visualization 3: Hourly Sales Pattern ────────────────────────────────────
hourly = df.groupby('hour_of_day').agg(count=('money','count'), avg_rev=('money','mean')).reset_index().sort_values('hour_of_day')

fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()
ax1.bar(hourly['hour_of_day'], hourly['count'], color='#3b82d4', alpha=0.75, label='Transaction Count', width=0.6)
ax2.plot(hourly['hour_of_day'], hourly['avg_rev'], color='#f97316', linewidth=2.5, marker='o', markersize=5, label='Avg Revenue (CZK)')
ax1.set_xlabel('Hour of Day (24h)', fontsize=11)
ax1.set_ylabel('Transaction Count', fontsize=11, color='#3b82d4')
ax2.set_ylabel('Avg Revenue (CZK)', fontsize=11, color='#f97316')
ax1.set_xticks(hourly['hour_of_day'])
ax1.set_title('Visualization 3 — Hourly Sales Pattern', fontsize=13, fontweight='bold')
peak_row = hourly.loc[hourly['count'].idxmax()]
ax1.annotate(f'Peak: {int(peak_row["hour_of_day"])}h', xy=(peak_row['hour_of_day'], peak_row['count']),
             xytext=(peak_row['hour_of_day']+0.5, peak_row['count']+5), fontsize=8, color='navy',
             arrowprops=dict(arrowstyle='->', color='navy', lw=1.2))
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'viz3_hourly_pattern.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak hour: {int(peak_row["hour_of_day"])}:00 with {int(peak_row["count"])} transactions')

In [ ]:
# ── Visualization 4: Weekday Sales Distribution ──────────────────────────────
wd_order = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7}
weekday = df.groupby(['Weekday','Weekdaysort']).agg(count=('money','count'), avg_rev=('money','mean')).reset_index().sort_values('Weekdaysort')

x = np.arange(len(weekday))
fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()
ax1.bar(x - 0.2, weekday['count'], 0.4, color='#3b82d4', alpha=0.8, label='Transactions')
ax2.bar(x + 0.2, weekday['avg_rev'], 0.4, color='#7c5cd8', alpha=0.8, label='Avg Revenue (CZK)')
ax1.set_xticks(x)
ax1.set_xticklabels(weekday['Weekday'], fontsize=11)
ax1.set_ylabel('Transaction Count', fontsize=11, color='#3b82d4')
ax2.set_ylabel('Avg Revenue (CZK)', fontsize=11, color='#7c5cd8')
ax1.set_title('Visualization 4 — Weekday Sales Distribution', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'viz4_weekday_pattern.png'), dpi=150, bbox_inches='tight')
plt.show()
busiest = weekday.loc[weekday['count'].idxmax()]
print(f'Busiest day: {busiest["Weekday"]} ({int(busiest["count"]):,} transactions)')

In [ ]:
# ── Visualization 5: Monthly Revenue Trend ───────────────────────────────────
df2 = df.copy()
df2['year_month'] = df2['Date'].dt.to_period('M')
monthly = df2.groupby('year_month').agg(total_revenue=('money','sum'), count=('money','count')).reset_index().sort_values('year_month')
monthly['label'] = monthly['year_month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()
ax1.fill_between(range(len(monthly)), monthly['total_revenue'], alpha=0.25, color='#16a34a')
ax1.plot(range(len(monthly)), monthly['total_revenue'], color='#16a34a', linewidth=2.5, marker='o', markersize=5, label='Total Revenue (CZK)')
ax2.bar(range(len(monthly)), monthly['count'], color='#3b82d4', alpha=0.3, label='Transactions', width=0.5)
ax1.set_xticks(range(len(monthly)))
ax1.set_xticklabels(monthly['label'], rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('Total Revenue (CZK)', fontsize=11, color='#16a34a')
ax2.set_ylabel('Transaction Count', fontsize=11, color='#3b82d4')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax1.set_title('Visualization 5 — Monthly Revenue & Transaction Trend (13 Months)', fontsize=13, fontweight='bold')
peak_idx = monthly['total_revenue'].idxmax()
ax1.annotate(f'Peak: {monthly["label"].iloc[peak_idx]}',
             xy=(peak_idx, monthly['total_revenue'].iloc[peak_idx]),
             xytext=(peak_idx-1, monthly['total_revenue'].iloc[peak_idx]*0.93),
             fontsize=8, color='darkgreen', arrowprops=dict(arrowstyle='->', color='darkgreen', lw=1.2))
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'viz5_monthly_trend.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak revenue month: {monthly["label"].iloc[peak_idx]} ({monthly["total_revenue"].iloc[peak_idx]:,.2f} CZK)')

## Section 11 — Exactly 5 Observations

Each observation is a direct, factual statement from the data — no interpretation.

In [ ]:
# Compute values for all 5 observations programmatically
prod_counts = df['coffee_name'].value_counts()
top1_name = prod_counts.index[0]
top1_count = prod_counts.iloc[0]
top1_pct = top1_count / total_tx * 100

hourly_counts = df['hour_of_day'].value_counts()
peak_h = hourly_counts.index[0]
peak_h_count = hourly_counts.iloc[0]
peak_h_pct = peak_h_count / total_tx * 100

tod_counts = df['Time_of_Day'].value_counts()
top_tod = tod_counts.index[0]
top_tod_pct = tod_counts.iloc[0] / total_tx * 100

wd_counts = df['Weekday'].value_counts()
top_wd = wd_counts.index[0]

rev_by_product = df.groupby('coffee_name')['money'].sum()
top_rev_product = rev_by_product.idxmax()
top_rev_pct = rev_by_product.max() / rev_by_product.sum() * 100

observations = [
    f"Observation 1 — Product Popularity: '{top1_name}' is the most-ordered product with "
    f"{top1_count:,} transactions ({top1_pct:.1f}% of all sales).",
    
    f"Observation 2 — Peak Hour: Hour {peak_h}:00 records the highest transaction volume "
    f"({peak_h_count:,} transactions, {peak_h_pct:.1f}% of total).",
    
    f"Observation 3 — Time Period: '{top_tod}' period accounts for the highest share of "
    f"transactions ({top_tod_pct:.1f}%).",
    
    f"Observation 4 — Revenue Concentration: '{top_rev_product}' generates the highest "
    f"total revenue ({top_rev_pct:.1f}% of total revenue).",
    
    f"Observation 5 — Payment Exclusivity: 100% of {total_tx:,} transactions in the "
    f"dataset use card payment. No cash transactions were recorded during the 13-month period.",
]

for obs in observations:
    print(obs)
    print()

## Section 12 — Exactly 5 Business Insights

Each insight connects an observation to a business implication.

In [ ]:
insights = [
    "Insight 1 — Inventory Priority: The dominance of a few coffee products (Latte, "
    "Americano with Milk) means that running out of their ingredients carries disproportionate "
    "revenue risk. Safety stock for these products should be larger than for niche items.",
    
    "Insight 2 — Staffing Windows: Concentrated transaction peaks at specific hours indicate "
    "that deploying maximum staff (baristas, cashiers) during these windows would reduce "
    "customer wait time and improve service quality during high-demand periods.",
    
    "Insight 3 — Seasonal Revenue Planning: Monthly revenue variation across the 13-month "
    "window shows non-trivial fluctuation. Lower-revenue months may benefit from targeted "
    "promotions or limited-time product offers to maintain consistent cash flow.",
    
    "Insight 4 — Weekend Strategy: Weekday-level analysis reveals whether weekdays or "
    "weekends drive more volume. If weekends show higher per-session revenue, extended "
    "weekend hours or special weekend menus could amplify this naturally higher spend.",
    
    "Insight 5 — Digital-First Operation: The complete absence of cash transactions signals "
    "that this business already operates on a fully digital payment infrastructure. This "
    "is an ideal foundation for integrating CRM, loyalty points, and customer analytics.",
]

for ins in insights:
    print(ins)
    print()

## Section 13 — Exactly 3 Hypotheses

In [ ]:
hypotheses = [
    {
        'id': 'H1',
        'title': 'Transaction Counts Differ Across Time-of-Day',
        'H0': 'The distribution of daily transaction counts is the same across Morning, Afternoon, and Night.',
        'H1': 'At least one Time_of_Day period has a significantly different distribution.',
        'test': 'Kruskal-Wallis H-test (α = 0.05)',
    },
    {
        'id': 'H2',
        'title': 'Transaction Amounts Differ Across Coffee Products',
        'H0': 'The distribution of transaction amounts is the same across all 8 coffee products.',
        'H1': 'At least one product has a significantly different price distribution.',
        'test': 'Kruskal-Wallis H-test (α = 0.05)',
    },
    {
        'id': 'H3',
        'title': 'Daily Transaction Volume Differs Across Weekdays',
        'H0': 'The distribution of daily transaction counts is the same across all weekdays.',
        'H1': 'At least one weekday shows significantly different daily transaction volume.',
        'test': 'Kruskal-Wallis H-test (α = 0.05)',
    },
]

for h in hypotheses:
    print(f"{h['id']}: {h['title']}")
    print(f"  H0: {h['H0']}")
    print(f"  H1: {h['H1']}")
    print(f"  Test: {h['test']}")
    print()

## Section 14 — Statistical Testing

In [ ]:
alpha = 0.05

# ── H1: Time-of-Day ──────────────────────────────────────────────────────────
daily_tod = df.groupby(['Date', 'Time_of_Day']).size().reset_index(name='count')
tod_groups = {t: daily_tod[daily_tod['Time_of_Day'] == t]['count'].values
              for t in daily_tod['Time_of_Day'].unique()}
h1_stat, h1_p = stats.kruskal(*[v for v in tod_groups.values() if len(v) > 1])

# ── H2: Coffee Product Amounts ───────────────────────────────────────────────
coffee_groups = {c: df[df['coffee_name'] == c]['money'].values for c in df['coffee_name'].unique()}
h2_stat, h2_p = stats.kruskal(*coffee_groups.values())

# ── H3: Weekday daily counts ─────────────────────────────────────────────────
daily_wd = df.groupby(['Date', 'Weekday', 'Weekdaysort']).size().reset_index(name='count')
wd_groups = {w: daily_wd[daily_wd['Weekday'] == w]['count'].values for w in daily_wd['Weekday'].unique()}
h3_stat, h3_p = stats.kruskal(*[v for v in wd_groups.values() if len(v) > 1])

# Print results table
results = pd.DataFrame([
    {'Hypothesis': 'H1: Time-of-Day vs Tx Count', 'Test': 'Kruskal-Wallis', 'Statistic': round(h1_stat, 4),
     'p-value': round(h1_p, 6), 'Decision': 'Reject H0' if h1_p < alpha else 'Fail to Reject H0'},
    {'Hypothesis': 'H2: Product vs Amount',       'Test': 'Kruskal-Wallis', 'Statistic': round(h2_stat, 4),
     'p-value': round(h2_p, 6), 'Decision': 'Reject H0' if h2_p < alpha else 'Fail to Reject H0'},
    {'Hypothesis': 'H3: Weekday vs Daily Count',  'Test': 'Kruskal-Wallis', 'Statistic': round(h3_stat, 4),
     'p-value': round(h3_p, 6), 'Decision': 'Reject H0' if h3_p < alpha else 'Fail to Reject H0'},
])

print('HYPOTHESIS TEST RESULTS (α = 0.05):')
print(results.to_string(index=False))
print()
print('NOTE: Statistical significance does NOT imply causation.')
print('Kruskal-Wallis is non-parametric; does not assume normality.')

## Section 15 — Exactly 3 Recommendations

In [ ]:
print("""RECOMMENDATION 1 — Prioritize Inventory for Top-Selling Products
Finding: A small number of products (Latte, Americano with Milk) account for a disproportionate
         share of both transaction volume and total revenue.
Action:  Implement tiered inventory management — maintain higher safety stock levels and shorter
         replenishment cycles for the top 3 products.
Data Support: Product transaction count and revenue analysis shows clear concentration.
Expected Benefit: Reduces stockout risk for high-revenue items; prevents lost sales.
Trade-off: Slightly higher working capital tied up in fast-moving inventory.
"""
)

print("""RECOMMENDATION 2 — Peak-Hour Staff Scheduling
Finding: Transaction volume is concentrated at specific hours of the day.
Action:  Schedule maximum staff (baristas + cashiers) during the 2-3 peak hours;
         consider staggered shift starts to reduce off-peak idle time.
Data Support: Hourly analysis shows clear transaction peaks at specific hours.
Expected Benefit: Shorter wait times, higher customer throughput during peaks.
Trade-off: May require part-time staff arrangements for peak-only coverage.
"""
)

print("""RECOMMENDATION 3 — Monthly Promotional Planning for Low-Revenue Months
Finding: Monthly revenue varies significantly across the 13-month window.
Action:  Identify the 3 lowest-revenue calendar months from historical data and design
         pre-planned promotional campaigns (bundle deals, limited-time items) to activate
         in those months each year.
Data Support: Monthly trend analysis shows clear revenue dips in specific months.
Expected Benefit: Smooths revenue curve; reduces seasonal revenue risk.
Trade-off: Promotions may marginally reduce per-transaction margin.
"""
)

## Section 16 — Predictive Problem Definition

In [ ]:
print("""SELECTED PREDICTIVE TASK: Daily Revenue Forecasting (Regression)

RATIONALE FOR TASK SELECTION:
- 'money' per transaction is largely determined by coffee_name + price_tier.
  Predicting it would be nearly trivial — no genuine learning required.
- Daily revenue forecasting is operationally meaningful: the business can use
  predictions for staffing, inventory, and cash-flow planning.
- The dataset covers 13 months (398 active days after aggregation), sufficient
  for time-series regression with lag/rolling features.

TARGET VARIABLE: revenue (daily sum of money)
FEATURES: lag features, rolling averages, calendar features (no future data used)
SPLIT: Chronological — 80% earliest dates for training, 20% latest for testing.
""")

print(f'Total active sales days: {daily.shape[0]}')
split_idx = int(daily.shape[0] * 0.80)
print(f'Training days: {split_idx} ({daily["Date"].iloc[0].date()} → {daily["Date"].iloc[split_idx-1].date()})')
print(f'Test days:     {daily.shape[0] - split_idx} ({daily["Date"].iloc[split_idx].date()} → {daily["Date"].iloc[-1].date()})')

## Section 17 — Leakage Audit

In [ ]:
leakage_audit = pd.DataFrame([
    ['revenue_lag_1',        'Previous day revenue',                   'None',              'Include'],
    ['revenue_lag_7',        'Same weekday last week revenue',         'None',              'Include'],
    ['revenue_rolling_7',    '7-day rolling mean (shift=1 applied)',   'None',              'Include'],
    ['revenue_rolling_14',   '14-day rolling mean (shift=1 applied)',  'None',              'Include'],
    ['day_of_week',          'Weekday of target date',                 'None (calendar)',   'Include'],
    ['month',                'Month of target date',                   'None (calendar)',   'Include'],
    ['is_weekend',           'Sat/Sun flag for target date',           'None (calendar)',   'Include'],
    ['current_day_revenue',  'The target variable itself',             'CRITICAL LEAKAGE',  'EXCLUDED'],
    ['coffee_name (raw)',    'Transaction-level product detail',       'HIGH LEAKAGE',      'EXCLUDED'],
], columns=['Feature', 'Relationship to Target', 'Leakage Risk', 'ML Decision'])

print('FEATURE LEAKAGE AUDIT:')
leakage_audit

## Section 18 — Train/Test Strategy

In [ ]:
# Build daily features with lags
daily_feat = daily.copy()
daily_feat['day_of_week']   = daily_feat['Date'].dt.dayofweek
daily_feat['day_of_month']  = daily_feat['Date'].dt.day
daily_feat['week_of_year']  = daily_feat['Date'].dt.isocalendar().week.astype(int)
daily_feat['month']         = daily_feat['Date'].dt.month
daily_feat['quarter']       = daily_feat['Date'].dt.quarter
daily_feat['is_weekend']    = (daily_feat['day_of_week'] >= 5).astype(int)

for lag in [1, 2, 3, 7, 14]:
    daily_feat[f'revenue_lag_{lag}']    = daily_feat['revenue'].shift(lag)
    daily_feat[f'count_lag_{lag}']      = daily_feat['transaction_count'].shift(lag)

for window in [7, 14, 30]:
    daily_feat[f'revenue_rolling_{window}'] = daily_feat['revenue'].shift(1).rolling(window, min_periods=1).mean().round(2)
    daily_feat[f'count_rolling_{window}']   = daily_feat['transaction_count'].shift(1).rolling(window, min_periods=1).mean().round(2)

daily_feat = daily_feat.dropna().reset_index(drop=True)

feature_cols = [c for c in daily_feat.columns if c not in ['Date', 'revenue', 'transaction_count']]
target_col = 'revenue'

n = len(daily_feat)
split_idx = int(n * 0.80)
train = daily_feat.iloc[:split_idx]
test  = daily_feat.iloc[split_idx:]

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')
print(f'\nTrain: {len(train)} days  ({train["Date"].min().date()} → {train["Date"].max().date()})')
print(f'Test:  {len(test)} days  ({test["Date"].min().date()} → {test["Date"].max().date()})')
print('\nChronological split — NO data shuffling performed.')

## Section 19 — Model Training

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

X_train = train[feature_cols].values
y_train = train[target_col].values
X_test  = test[feature_cols].values
y_test  = test[target_col].values

models = {
    'Ridge':              Ridge(alpha=1.0),
    'Decision Tree':      DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=8),
    'Random Forest':      RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE),
    'HistGradBoost':      HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE),
}

print('Training models on chronological train split...')
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'  {name}: trained')

## Section 20 — Model Comparison

In [ ]:
rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    mask = y_test != 0
    mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100 if mask.sum() else float('nan')
    rows.append({'Model': name, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R²': round(r2,4), 'MAPE%': round(mape,2)})

comparison_df = pd.DataFrame(rows).sort_values('MAE')
print('=== MODEL COMPARISON (Test Set) ===')
print(comparison_df.to_string(index=False))
best_model_name = comparison_df.iloc[0]['Model']
print(f'\nBest model by MAE: {best_model_name}')

## Section 21 — Model Evaluation

In [ ]:
# Use Random Forest as primary model
primary_model = models['Random Forest']
y_pred_rf = primary_model.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print('=== Random Forest — Test Set Evaluation ===')
print(f'MAE:   {mae_rf:.2f} CZK  (avg absolute prediction error)')
print(f'RMSE:  {rmse_rf:.2f} CZK  (penalizes larger errors more)')
print(f'R²:    {r2_rf:.4f}    (proportion of variance explained)')

# Actual vs predicted plot
fig, ax = plt.subplots(figsize=(12, 5))
test_dates = test['Date'].values
ax.plot(range(len(y_test)), y_test, color='#16a34a', linewidth=2, label='Actual Revenue', marker='o', markersize=3)
ax.plot(range(len(y_pred_rf)), y_pred_rf, color='#3b82d4', linewidth=2, linestyle='--', label='Predicted Revenue', marker='s', markersize=3)
ax.fill_between(range(len(y_test)), y_test, y_pred_rf, alpha=0.1, color='orange')
ax.set_xlabel('Test Day Index', fontsize=11)
ax.set_ylabel('Revenue (CZK)', fontsize=11)
ax.set_title(f'Random Forest — Actual vs Predicted Daily Revenue (Test Set)\nMAE={mae_rf:.2f}  RMSE={rmse_rf:.2f}  R²={r2_rf:.4f}', fontsize=12, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'model_actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 22 — Prediction Results

In [ ]:
pred_df = test[['Date', 'revenue']].copy()
pred_df['predicted_revenue'] = y_pred_rf.round(2)
pred_df['absolute_error']    = np.abs(pred_df['revenue'] - pred_df['predicted_revenue']).round(2)
pred_df['pct_error']         = (pred_df['absolute_error'] / pred_df['revenue'] * 100).round(1)
print('Sample Predictions (first 15 test dates):')
pred_df.head(15)

## Section 23 — Feature Importance

In [ ]:
fi = pd.Series(primary_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top_fi = fi.head(12)
colors = ['#3b82d4'] * len(top_fi)
bars = ax.barh(top_fi.index[::-1], top_fi.values[::-1], color=colors, edgecolor='white')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.002, bar.get_y() + bar.get_height()/2, f'{w:.4f}', va='center', fontsize=8)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_title('Random Forest — Feature Importance (Top 12)\n'
             'Note: Importance reflects model association, not causation.', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 most important features:')
for f, v in fi.head(5).items():
    print(f'  {f}: {v:.4f}')

## Section 24 — Final Conclusion

In [ ]:
print(f"""=== FINAL PROJECT SUMMARY ===

DATASET:
  - {total_tx:,} transactions across {active_days} active sales days
  - Date range: {df['Date'].min().date()} to {df['Date'].max().date()} (13 months)
  - 8 distinct coffee products, all card payments
  - Prices evolved across 3 distinct tiers during the observation period

EDA FINDINGS:
  - Top product: {top1_name} ({top1_count:,} tx, {top1_pct:.1f}% of all sales)
  - Peak hour: {peak_h}:00 ({peak_h_count:,} transactions)
  - Peak time period: {top_tod} ({top_tod_pct:.1f}% of total)
  - Top revenue product: {top_rev_product} ({top_rev_pct:.1f}% of revenue)

STATISTICAL TESTING:
  - H1 (Time-of-Day): {'Rejected' if h1_p < alpha else 'Not rejected'} (p={h1_p:.6f})
  - H2 (Product amounts): {'Rejected' if h2_p < alpha else 'Not rejected'} (p={h2_p:.6f})
  - H3 (Weekday counts): {'Rejected' if h3_p < alpha else 'Not rejected'} (p={h3_p:.6f})

MACHINE LEARNING:
  - Task: Daily Revenue Forecasting
  - Best model: Random Forest Regressor
  - Test MAE:  {mae_rf:.2f} CZK
  - Test RMSE: {rmse_rf:.2f} CZK
  - Test R²:   {r2_rf:.4f}
  - Split: Chronological 80/20 — no data leakage

NODE.JS DASHBOARD:
  - Interactive filters: coffee, time, weekday, month, date range
  - 8 dynamic KPI cards + 7 interactive charts
  - Transaction table with pagination
  - Real-time revenue prediction interface
  - Architecture: Browser → Node.js (3000) → Python FastAPI (8000)
""")